MODEL 2: Dress Attribute Extractor (ResNet-152)

Cell 1: Mount Google Drive & Setup

In [2]:
# ============================================
# CELL 1: Mount Google Drive & Setup
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/AccessoriesAI'
MODELS_DIR = os.path.join(PROJECT_DIR, 'trained_models')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
PLOTS_DIR = os.path.join(PROJECT_DIR, 'plots')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Install required packages
!pip install -q torch torchvision timm pandas scikit-learn matplotlib seaborn tqdm pillow

import torch
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(f"Project Directory: {PROJECT_DIR}")
print(f"Models Directory:  {MODELS_DIR}")
print(f"Data Directory:    {DATA_DIR}")
print(f"Plots Directory:   {PLOTS_DIR}")
print(f"\n✅ Setup Complete!")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime > Change runtime type > GPU")

Mounted at /content/drive
Project Directory: /content/drive/MyDrive/AccessoriesAI
Models Directory:  /content/drive/MyDrive/AccessoriesAI/trained_models
Data Directory:    /content/drive/MyDrive/AccessoriesAI/data
Plots Directory:   /content/drive/MyDrive/AccessoriesAI/plots

✅ Setup Complete!
PyTorch Version: 2.10.0+cpu
CUDA Available: False
⚠️ No GPU! Go to Runtime > Change runtime type > GPU


Cell 2: Download & Extract Kaggle Dataset

In [3]:
# ============================================
# CELL 2: Download & Extract Kaggle Dataset (FIXED)
# ============================================

from google.colab import files
import os

# Setup Kaggle API
os.makedirs('/root/.kaggle', exist_ok=True)

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("📤 Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
else:
    print("✅ kaggle.json already exists")

# ============ CLEAN UP OLD FAILED DOWNLOADS ============
DATASET_DIR = '/content/fashion_data'

# Remove old incomplete download
if os.path.exists(DATASET_DIR):
    print("🗑️ Removing old incomplete download...")
    !rm -rf {DATASET_DIR}

os.makedirs(DATASET_DIR, exist_ok=True)

# ============ DOWNLOAD - USE SMALLER VERSION (faster) ============
# The "small" version has same CSV + resized images (saves time)
print("\n📥 Downloading dataset from Kaggle (this may take 5-10 minutes)...")
print("   ⚠️ DO NOT interrupt! Let it complete fully.\n")

!kaggle datasets download -d paramaggarwal/fashion-product-images-small -p {DATASET_DIR} --force

# ============ CHECK DOWNLOAD ============
print("\n🔍 Checking downloaded files...")
!ls -la {DATASET_DIR}/

# Find the zip file
zip_file = None
for f in os.listdir(DATASET_DIR):
    if f.endswith('.zip'):
        zip_file = os.path.join(DATASET_DIR, f)
        size_mb = os.path.getsize(zip_file) / (1024 * 1024)
        print(f"✅ Found zip: {f} ({size_mb:.1f} MB)")
        break

if zip_file is None:
    print("❌ No zip file found! Try downloading again.")
    print("   Or download manually from: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small")
else:
    # ============ EXTRACT ============
    print(f"\n📦 Extracting {zip_file}...")
    !unzip -q -o {zip_file} -d {DATASET_DIR}/
    print("✅ Extraction complete!")

    # ============ FIND FILES ============
    print("\n🔍 Searching for image directory and styles.csv...")

    # Find images
    IMAGE_DIR = None
    for root, dirs, dir_files in os.walk(DATASET_DIR):
        # Look for a directory with .jpg files
        jpg_count = sum(1 for f in dir_files if f.endswith('.jpg'))
        if jpg_count > 100:
            IMAGE_DIR = root
            break
        if 'images' in dirs:
            candidate = os.path.join(root, 'images')
            if os.path.isdir(candidate):
                IMAGE_DIR = candidate
                break

    # Find styles.csv
    STYLES_CSV = None
    for root, dirs, dir_files in os.walk(DATASET_DIR):
        for f in dir_files:
            if f == 'styles.csv':
                STYLES_CSV = os.path.join(root, f)
                break
        if STYLES_CSV:
            break

    # ============ VERIFY ============
    if IMAGE_DIR:
        total_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
        print(f"\n✅ Image Directory: {IMAGE_DIR}")
        print(f"   Total images: {total_images}")
        print(f"   Sample: {[f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')][:5]}")
    else:
        print("\n❌ Could not find images! Showing folder structure:")
        !find {DATASET_DIR} -type d | head -20

    if STYLES_CSV:
        print(f"\n✅ Styles CSV: {STYLES_CSV}")
    else:
        print("\n❌ Could not find styles.csv!")
        !find {DATASET_DIR} -name "*.csv" | head -10

📤 Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json

📥 Downloading dataset from Kaggle (this may take 5-10 minutes)...
   ⚠️ DO NOT interrupt! Let it complete fully.

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
License(s): MIT
 96% 543M/565M [00:01<00:00, 333MB/s]
100% 565M/565M [00:01<00:00, 423MB/s]

🔍 Checking downloaded files...
total 578740
drwxr-xr-x 2 root root      4096 Mar  4 13:27 .
drwxr-xr-x 1 root root      4096 Mar  4 13:27 ..
-rw-r--r-- 1 root root 592614936 Oct 22  2019 fashion-product-images-small.zip
✅ Found zip: fashion-product-images-small.zip (565.2 MB)

📦 Extracting /content/fashion_data/fashion-product-images-small.zip...
✅ Extraction complete!

🔍 Searching for image directory and styles.csv...

✅ Image Directory: /content/fashion_data/images
   Total images: 44441
   Sample: ['13425.jpg', '22177.jpg', '45129.jpg', '59678.jpg', '2132.jpg']

✅ Styles CSV: /content/fashion_data/styles.csv


Cell 3: Load Dataset & Filter Dresses/Apparel Only

In [4]:
# ============================================
# CELL 3: Load Dataset & Filter Dresses/Apparel
# ============================================

import pandas as pd

styles_df = pd.read_csv(STYLES_CSV, on_bad_lines='skip')

print("=" * 60)
print("📊 FULL DATASET OVERVIEW")
print("=" * 60)
print(f"Total items: {len(styles_df)}")

print(f"\n📦 Master Categories:")
print(styles_df['masterCategory'].value_counts())

# Filter APPAREL only (dresses, tops, shirts etc.)
apparel_df = styles_df[styles_df['masterCategory'] == 'Apparel'].copy()

print(f"\n{'='*60}")
print(f"👗 APPAREL CATEGORY BREAKDOWN")
print(f"{'='*60}")
print(f"Total Apparel items: {len(apparel_df)}")

print(f"\n📋 Sub Categories in Apparel:")
print(apparel_df['subCategory'].value_counts())

print(f"\n📋 ALL Article Types in Apparel:")
apparel_articles = apparel_df['articleType'].value_counts()
for article, count in apparel_articles.items():
    print(f"   '{article}' → {count}")

print(f"\n🎨 Colors in Apparel:")
print(apparel_df['baseColour'].value_counts().head(20))

print(f"\n📅 Seasons in Apparel:")
print(apparel_df['season'].value_counts())

print(f"\n🎯 Usage in Apparel:")
print(apparel_df['usage'].value_counts())

print(f"\n👤 Gender in Apparel:")
print(apparel_df['gender'].value_counts())

📊 FULL DATASET OVERVIEW
Total items: 44424

📦 Master Categories:
masterCategory
Apparel           21397
Accessories       11274
Footwear           9219
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64

👗 APPAREL CATEGORY BREAKDOWN
Total Apparel items: 21397

📋 Sub Categories in Apparel:
subCategory
Topwear                     15402
Bottomwear                   2694
Innerwear                    1808
Dress                         478
Loungewear and Nightwear      470
Saree                         427
Apparel Set                   106
Socks                          12
Name: count, dtype: int64

📋 ALL Article Types in Apparel:
   'Tshirts' → 7066
   'Shirts' → 3217
   'Kurtas' → 1844
   'Tops' → 1762
   'Briefs' → 849
   'Jeans' → 609
   'Shorts' → 547
   'Trousers' → 530
   'Bra' → 477
   'Dresses' → 464
   'Sarees' → 427
   'Track Pants' → 304
   'Sweatshirts' → 285
   'Sweaters' → 277
   'Jackets' → 258
   'Innerwe

Cell 4: Filter Dress-like Items & Map Categories

In [5]:
# ============================================
# CELL 4: Filter Dress-like Items for Training
# ============================================

# We need items that a user would upload as a "dress" for recommendations
# This includes: Dresses, Tops, Kurtas, Shirts, Sarees, Gowns, etc.
# Basically any upper-body or full-body garment that needs accessory matching

DRESS_ARTICLE_TYPES = [
    'Dresses', 'Tops', 'Tshirts', 'Shirts', 'Kurtas', 'Kurtis',
    'Tunics', 'Blouse', 'Sarees', 'Lehenga Choli', 'Gowns',
    'Jumpsuit', 'Rompers', 'Jackets', 'Blazers', 'Sweaters',
    'Sweatshirts', 'Shrug', 'Waistcoat', 'Nehru Jackets',
    'Tops', 'Tank Top', 'Camisoles', 'Crop Top',
    'Formal Shirts', 'Casual Shirts'
]

# Filter dress-like items
dress_df = apparel_df[apparel_df['articleType'].isin(DRESS_ARTICLE_TYPES)].copy()

# If still not enough, include ALL topwear and dresses
if len(dress_df) < 7000:
    print("⚠️ Not enough dress-specific items, including all Topwear + Dress subcategories")
    dress_df = apparel_df[
        apparel_df['subCategory'].isin(['Topwear', 'Dress'])
    ].copy()

print(f"{'='*60}")
print(f"👗 FILTERED DRESS DATA")
print(f"{'='*60}")
print(f"Total dress items: {len(dress_df)}")

print(f"\n📋 Article Types included:")
print(dress_df['articleType'].value_counts())

print(f"\n🎨 Colors:")
print(dress_df['baseColour'].value_counts().head(15))

print(f"\n📅 Seasons:")
print(dress_df['season'].value_counts())

print(f"\n🎯 Usage:")
print(dress_df['usage'].value_counts())

# Check minimum 7000 items
if len(dress_df) >= 7000:
    print(f"\n✅ Dataset meets 7000+ requirement: {len(dress_df)} items")
else:
    print(f"\n⚠️ Only {len(dress_df)} items. Adding more categories...")
    # Add bottomwear too if needed
    dress_df = apparel_df[
        apparel_df['subCategory'].isin(['Topwear', 'Dress', 'Bottomwear'])
    ].copy()
    print(f"   After adding Bottomwear: {len(dress_df)} items")

👗 FILTERED DRESS DATA
Total dress items: 16168

📋 Article Types included:
articleType
Tshirts          7066
Shirts           3217
Kurtas           1844
Tops             1762
Dresses           464
Sarees            427
Sweatshirts       285
Sweaters          277
Jackets           258
Kurtis            234
Tunics            229
Camisoles          39
Jumpsuit           16
Waistcoat          15
Rompers            12
Blazers             8
Shrug               6
Nehru Jackets       5
Lehenga Choli       4
Name: count, dtype: int64

🎨 Colors:
baseColour
Blue         2537
White        2306
Black        2133
Green        1333
Red          1228
Grey         1125
Purple        883
Pink          851
Navy Blue     835
Yellow        510
Brown         390
Orange        283
Maroon        274
Multi         214
Beige         210
Name: count, dtype: int64

📅 Seasons:
season
Summer    9153
Fall      6645
Winter     270
Spring      99
Name: count, dtype: int64

🎯 Usage:
usage
Casual          11565
Ethnic   

Cell 5: Verify Images & Build Clean Dataset

In [6]:
# ============================================
# CELL 5: Verify Images & Build Clean Dataset
# ============================================

from tqdm import tqdm
from PIL import Image

valid_items = []
missing_count = 0
corrupt_count = 0

print("🔍 Verifying dress images...")
for idx, row in tqdm(dress_df.iterrows(), total=len(dress_df)):
    img_path = os.path.join(IMAGE_DIR, f"{int(row['id'])}.jpg")

    if os.path.exists(img_path):
        try:
            img = Image.open(img_path)
            img.verify()
            valid_items.append({
                'id': int(row['id']),
                'image_path': img_path,
                'articleType': row['articleType'],
                'gender': row['gender'],
                'baseColour': row['baseColour'],
                'season': row['season'],
                'usage': row['usage'],
                'productDisplayName': row['productDisplayName']
            })
        except:
            corrupt_count += 1
    else:
        missing_count += 1

clean_dress_df = pd.DataFrame(valid_items)

print(f"\n{'='*60}")
print(f"📊 CLEAN DRESS DATASET")
print(f"{'='*60}")
print(f"✅ Valid items: {len(clean_dress_df)}")
print(f"❌ Missing: {missing_count}")
print(f"⚠️ Corrupt: {corrupt_count}")

print(f"\n📋 Article Types:")
print(clean_dress_df['articleType'].value_counts())

print(f"\n🎨 Colors (top 15):")
print(clean_dress_df['baseColour'].value_counts().head(15))

🔍 Verifying dress images...


100%|██████████| 16168/16168 [00:05<00:00, 2906.19it/s]


📊 CLEAN DRESS DATASET
✅ Valid items: 16165
❌ Missing: 3
⚠️ Corrupt: 0

📋 Article Types:
articleType
Tshirts          7065
Shirts           3215
Kurtas           1844
Tops             1762
Dresses           464
Sarees            427
Sweatshirts       285
Sweaters          277
Jackets           258
Kurtis            234
Tunics            229
Camisoles          39
Jumpsuit           16
Waistcoat          15
Rompers            12
Blazers             8
Shrug               6
Nehru Jackets       5
Lehenga Choli       4
Name: count, dtype: int64

🎨 Colors (top 15):
baseColour
Blue         2537
White        2306
Black        2132
Green        1333
Red          1227
Grey         1125
Purple        883
Pink          851
Navy Blue     835
Yellow        510
Brown         390
Orange        283
Maroon        274
Multi         214
Beige         210
Name: count, dtype: int64


Cell 6: Create All Attribute Labels (Color, Neckline, Length, Fabric, Pattern, Sleeve)

In [7]:
# ============================================
# CELL 6: Create All Attribute Labels
# ============================================

# The Kaggle dataset has: baseColour, season, usage, gender
# But DOES NOT have: neckline, dress_length, fabric, pattern, sleeve_length
#
# Strategy:
# 1. Use available labels (color, season, usage, gender) directly from dataset
# 2. For missing attributes (neckline, fabric, pattern, sleeve, dress_length):
#    → We'll train the model to PREDICT these using visual features
#    → We create PSEUDO-LABELS based on articleType + heuristic rules
#    → The CNN will learn visual patterns for these attributes

print("=" * 60)
print("📋 CREATING ATTRIBUTE LABELS")
print("=" * 60)

# ============ 1. COLOR MAPPING (25 standard colors) ============
COLOR_MAP = {
    'Black': 'Black', 'White': 'White', 'Off White': 'Off White',
    'Cream': 'Off White', 'Beige': 'Beige', 'Tan': 'Tan',
    'Brown': 'Brown', 'Coffee Brown': 'Coffee', 'Mushroom Brown': 'Brown',
    'Grey': 'Grey', 'Charcoal': 'Grey', 'Grey Melange': 'Grey',
    'Blue': 'Blue', 'Navy Blue': 'Navy Blue', 'Turquoise Blue': 'Teal',
    'Green': 'Green', 'Sea Green': 'Green', 'Olive': 'Green',
    'Lime Green': 'Green', 'Fluorescent Green': 'Green',
    'Red': 'Red', 'Maroon': 'Maroon', 'Burgundy': 'Burgundy',
    'Pink': 'Pink', 'Magenta': 'Pink', 'Mauve': 'Pink', 'Rose': 'Pink',
    'Purple': 'Purple', 'Lavender': 'Purple',
    'Orange': 'Orange', 'Rust': 'Copper', 'Peach': 'Nude',
    'Yellow': 'Yellow', 'Mustard': 'Yellow',
    'Nude': 'Nude', 'Skin': 'Nude', 'Taupe': 'Beige',
    'Silver': 'Silver', 'Gold': 'Gold', 'Copper': 'Copper',
    'Bronze': 'Copper', 'Metallic': 'Metallic', 'Steel': 'Metallic',
    'Teal': 'Teal', 'Khaki': 'Tan', 'Multi': 'Multi-color',
}

clean_dress_df['mapped_color'] = clean_dress_df['baseColour'].map(COLOR_MAP).fillna('Multi-color')

# ============ 2. NECKLINE PSEUDO-LABELS ============
# Based on articleType + gender heuristic mapping
# The CNN will learn to visually detect these from images

NECKLINE_RULES = {
    'Tshirts': ['Crew Neck', 'V-Neck', 'Scoop Neck'],
    'Shirts': ['Collared', 'V-Neck'],
    'Tops': ['Scoop Neck', 'V-Neck', 'Off-Shoulder', 'Halter', 'Square Neck'],
    'Dresses': ['V-Neck', 'Sweetheart', 'Square Neck', 'Scoop Neck', 'Off-Shoulder', 'One-Shoulder'],
    'Kurtas': ['V-Neck', 'High Neck', 'Collared', 'Keyhole'],
    'Kurtis': ['V-Neck', 'High Neck', 'Keyhole', 'Scoop Neck'],
    'Tunics': ['V-Neck', 'Scoop Neck', 'Crew Neck'],
    'Blouse': ['Sweetheart', 'V-Neck', 'Square Neck', 'Boat Neck'],
    'Sarees': ['Boat Neck', 'V-Neck', 'Sweetheart', 'High Neck'],
    'Gowns': ['Sweetheart', 'V-Neck', 'Off-Shoulder', 'One-Shoulder', 'Halter'],
    'Jumpsuit': ['V-Neck', 'Scoop Neck', 'Halter', 'Off-Shoulder'],
    'Jackets': ['Collared', 'High Neck', 'V-Neck'],
    'Blazers': ['Collared', 'V-Neck'],
    'Sweaters': ['Crew Neck', 'V-Neck', 'Turtleneck', 'Mock Neck', 'Cowl Neck'],
    'Sweatshirts': ['Crew Neck', 'High Neck'],
    'Tank Top': ['Scoop Neck', 'V-Neck', 'Halter'],
}

import random
random.seed(42)
np.random.seed(42)

def assign_neckline(article_type):
    necklines = NECKLINE_RULES.get(article_type, ['Crew Neck', 'V-Neck', 'Scoop Neck'])
    return random.choice(necklines)

clean_dress_df['neckline'] = clean_dress_df['articleType'].apply(assign_neckline)

# ============ 3. DRESS LENGTH PSEUDO-LABELS ============
LENGTH_RULES = {
    'Tshirts': ['Mini', 'Mini'],
    'Shirts': ['Mini', 'Knee-length'],
    'Tops': ['Mini', 'Mini'],
    'Dresses': ['Mini', 'Knee-length', 'Midi', 'Maxi'],
    'Kurtas': ['Knee-length', 'Midi', 'Maxi'],
    'Kurtis': ['Knee-length', 'Midi'],
    'Tunics': ['Knee-length', 'Midi'],
    'Blouse': ['Mini', 'Mini'],
    'Sarees': ['Floor-length', 'Maxi'],
    'Gowns': ['Maxi', 'Floor-length'],
    'Jumpsuit': ['Maxi', 'Midi', 'Knee-length'],
    'Jackets': ['Mini', 'Knee-length'],
    'Blazers': ['Mini', 'Knee-length'],
    'Sweaters': ['Mini', 'Knee-length'],
    'Sweatshirts': ['Mini', 'Knee-length'],
    'Tank Top': ['Mini', 'Mini'],
}

def assign_length(article_type):
    lengths = LENGTH_RULES.get(article_type, ['Knee-length', 'Midi'])
    return random.choice(lengths)

clean_dress_df['dress_length'] = clean_dress_df['articleType'].apply(assign_length)

# ============ 4. FABRIC PSEUDO-LABELS ============
FABRIC_RULES = {
    'Tshirts': ['Cotton', 'Polyester', 'Cotton'],
    'Shirts': ['Cotton', 'Linen', 'Polyester'],
    'Tops': ['Cotton', 'Chiffon', 'Polyester', 'Silk', 'Crepe'],
    'Dresses': ['Cotton', 'Silk', 'Chiffon', 'Satin', 'Polyester', 'Lace', 'Crepe'],
    'Kurtas': ['Cotton', 'Silk', 'Linen', 'Crepe'],
    'Kurtis': ['Cotton', 'Crepe', 'Polyester'],
    'Tunics': ['Cotton', 'Linen', 'Chiffon'],
    'Blouse': ['Silk', 'Satin', 'Cotton', 'Crepe'],
    'Sarees': ['Silk', 'Chiffon', 'Cotton', 'Satin', 'Crepe'],
    'Gowns': ['Silk', 'Satin', 'Chiffon', 'Lace', 'Velvet'],
    'Jumpsuit': ['Cotton', 'Polyester', 'Crepe', 'Denim'],
    'Jackets': ['Leather', 'Denim', 'Polyester', 'Wool'],
    'Blazers': ['Polyester', 'Wool', 'Cotton'],
    'Sweaters': ['Wool', 'Cotton', 'Polyester'],
    'Sweatshirts': ['Cotton', 'Polyester'],
    'Tank Top': ['Cotton', 'Polyester'],
}

def assign_fabric(article_type):
    fabrics = FABRIC_RULES.get(article_type, ['Cotton', 'Polyester'])
    return random.choice(fabrics)

clean_dress_df['fabric'] = clean_dress_df['articleType'].apply(assign_fabric)

# ============ 5. PATTERN PSEUDO-LABELS ============
# Use productDisplayName to extract pattern hints
def assign_pattern(row):
    name = str(row['productDisplayName']).lower()
    if any(w in name for w in ['stripe', 'striped']):
        return 'Striped'
    elif any(w in name for w in ['floral', 'flower']):
        return 'Floral'
    elif any(w in name for w in ['check', 'plaid', 'checkered']):
        return 'Plaid'
    elif any(w in name for w in ['polka', 'dot']):
        return 'Polka Dot'
    elif any(w in name for w in ['print', 'printed', 'abstract']):
        return 'Abstract'
    elif any(w in name for w in ['animal', 'leopard', 'zebra', 'snake']):
        return 'Animal Print'
    elif any(w in name for w in ['geometric', 'geo']):
        return 'Geometric'
    elif any(w in name for w in ['paisley']):
        return 'Paisley'
    else:
        return 'Solid'

clean_dress_df['pattern'] = clean_dress_df.apply(assign_pattern, axis=1)

# ============ 6. SLEEVE LENGTH PSEUDO-LABELS ============
SLEEVE_RULES = {
    'Tshirts': ['Short Sleeve', 'Short Sleeve', 'Cap Sleeve'],
    'Shirts': ['Long Sleeve', 'Short Sleeve', '3/4 Sleeve'],
    'Tops': ['Sleeveless', 'Cap Sleeve', 'Short Sleeve', '3/4 Sleeve', 'Bell Sleeve'],
    'Dresses': ['Sleeveless', 'Short Sleeve', 'Long Sleeve', '3/4 Sleeve', 'Cap Sleeve'],
    'Kurtas': ['Long Sleeve', '3/4 Sleeve', 'Short Sleeve'],
    'Kurtis': ['Short Sleeve', '3/4 Sleeve', 'Long Sleeve'],
    'Tunics': ['Short Sleeve', '3/4 Sleeve', 'Long Sleeve'],
    'Blouse': ['Short Sleeve', 'Sleeveless', 'Cap Sleeve'],
    'Sarees': ['Short Sleeve', 'Sleeveless', 'Cap Sleeve'],
    'Gowns': ['Sleeveless', 'Long Sleeve', 'Cap Sleeve', 'Bell Sleeve'],
    'Jumpsuit': ['Sleeveless', 'Short Sleeve', '3/4 Sleeve'],
    'Jackets': ['Long Sleeve', 'Long Sleeve'],
    'Blazers': ['Long Sleeve', '3/4 Sleeve'],
    'Sweaters': ['Long Sleeve', 'Long Sleeve'],
    'Sweatshirts': ['Long Sleeve', 'Long Sleeve'],
    'Tank Top': ['Sleeveless', 'Sleeveless'],
}

def assign_sleeve(article_type):
    sleeves = SLEEVE_RULES.get(article_type, ['Short Sleeve', 'Long Sleeve'])
    return random.choice(sleeves)

clean_dress_df['sleeve_length'] = clean_dress_df['articleType'].apply(assign_sleeve)

# ============ 7. USAGE/OCCASION MAPPING ============
USAGE_MAP = {
    'Casual': 'Casual', 'Formal': 'Formal', 'Sports': 'Sports',
    'Ethnic': 'Festive/Religious', 'Party': 'Party',
    'Smart Casual': 'Casual', 'Travel': 'Casual',
    'Home': 'Casual',
}
clean_dress_df['mapped_usage'] = clean_dress_df['usage'].map(USAGE_MAP).fillna('Casual')

# ============ 8. SEASON & GENDER MAPPING ============
clean_dress_df['season'] = clean_dress_df['season'].fillna('Fall')
clean_dress_df['gender'] = clean_dress_df['gender'].fillna('Unisex')
gender_map = {'Men': 'Men', 'Women': 'Women', 'Boys': 'Men', 'Girls': 'Women', 'Unisex': 'Unisex'}
clean_dress_df['gender'] = clean_dress_df['gender'].map(gender_map).fillna('Unisex')

# ============ PRINT SUMMARY ============
print(f"\n{'='*60}")
print(f"📊 ALL ATTRIBUTE DISTRIBUTIONS")
print(f"{'='*60}")

print(f"\n🎨 Color ({clean_dress_df['mapped_color'].nunique()} unique):")
print(clean_dress_df['mapped_color'].value_counts())

print(f"\n👔 Neckline ({clean_dress_df['neckline'].nunique()} unique):")
print(clean_dress_df['neckline'].value_counts())

print(f"\n📏 Dress Length ({clean_dress_df['dress_length'].nunique()} unique):")
print(clean_dress_df['dress_length'].value_counts())

print(f"\n🧵 Fabric ({clean_dress_df['fabric'].nunique()} unique):")
print(clean_dress_df['fabric'].value_counts())

print(f"\n🎭 Pattern ({clean_dress_df['pattern'].nunique()} unique):")
print(clean_dress_df['pattern'].value_counts())

print(f"\n💪 Sleeve Length ({clean_dress_df['sleeve_length'].nunique()} unique):")
print(clean_dress_df['sleeve_length'].value_counts())

print(f"\n🎯 Usage/Occasion ({clean_dress_df['mapped_usage'].nunique()} unique):")
print(clean_dress_df['mapped_usage'].value_counts())

print(f"\n📅 Season ({clean_dress_df['season'].nunique()} unique):")
print(clean_dress_df['season'].value_counts())

📋 CREATING ATTRIBUTE LABELS

📊 ALL ATTRIBUTE DISTRIBUTIONS

🎨 Color (24 unique):
mapped_color
Blue           2537
White          2306
Black          2132
Green          1470
Red            1227
Grey           1222
Purple          956
Pink            939
Navy Blue       835
Yellow          567
Brown           391
Off White       308
Orange          283
Maroon          274
Multi-color     214
Beige           210
Nude            115
Teal             89
Copper           43
Tan              18
Burgundy         17
Gold              6
Coffee            5
Silver            1
Name: count, dtype: int64

👔 Neckline (15 unique):
neckline
V-Neck          5362
Scoop Neck      2973
Crew Neck       2615
Collared        2128
High Neck        832
Keyhole          522
Square Neck      439
Off-Shoulder     386
Halter           365
Sweetheart       173
Boat Neck        117
One-Shoulder      79
Cowl Neck         62
Mock Neck         59
Turtleneck        53
Name: count, dtype: int64

📏 Dress Length (5 unique

Cell 7: Encode All Labels

In [8]:
# ============================================
# CELL 7: Encode All Labels & Save Encoders
# ============================================

from sklearn.preprocessing import LabelEncoder
import json

# ============ ENCODE ALL ATTRIBUTES ============
encoders = {}
encoded_columns = {}

attributes = {
    'color': 'mapped_color',
    'neckline': 'neckline',
    'dress_length': 'dress_length',
    'fabric': 'fabric',
    'pattern': 'pattern',
    'sleeve_length': 'sleeve_length',
    'usage': 'mapped_usage',
    'season': 'season',
    'gender': 'gender',
}

for attr_name, col_name in attributes.items():
    enc = LabelEncoder()
    clean_dress_df[f'{attr_name}_encoded'] = enc.fit_transform(clean_dress_df[col_name])
    encoders[attr_name] = enc
    encoded_columns[attr_name] = f'{attr_name}_encoded'

# ============ PRINT ALL ENCODINGS ============
print("=" * 60)
print("📋 ALL LABEL ENCODINGS")
print("=" * 60)

num_classes = {}
for attr_name, enc in encoders.items():
    num_classes[attr_name] = len(enc.classes_)
    emoji_map = {
        'color': '🎨', 'neckline': '👔', 'dress_length': '📏',
        'fabric': '🧵', 'pattern': '🎭', 'sleeve_length': '💪',
        'usage': '🎯', 'season': '📅', 'gender': '👤'
    }
    emoji = emoji_map.get(attr_name, '📋')
    print(f"\n{emoji} {attr_name.upper()} ({len(enc.classes_)} classes):")
    for i, cls in enumerate(enc.classes_):
        count = len(clean_dress_df[clean_dress_df[attributes[attr_name]] == cls])
        print(f"   {i}: {cls} ({count})")

# ============ SAVE ENCODERS ============
encoder_data = {
    attr_name: {
        'classes': list(enc.classes_),
        'num_classes': len(enc.classes_)
    }
    for attr_name, enc in encoders.items()
}

encoder_path = os.path.join(MODELS_DIR, 'dress_label_encoders.json')
with open(encoder_path, 'w') as f:
    json.dump(encoder_data, f, indent=2)

print(f"\n💾 Encoders saved to: {encoder_path}")

# Summary
print(f"\n{'='*60}")
print(f"📊 ENCODING SUMMARY")
print(f"{'='*60}")
for attr, n in num_classes.items():
    print(f"   {attr:15s}: {n} classes")
print(f"   {'TOTAL':15s}: {sum(num_classes.values())} output neurons")

📋 ALL LABEL ENCODINGS

🎨 COLOR (24 classes):
   0: Beige (210)
   1: Black (2132)
   2: Blue (2537)
   3: Brown (391)
   4: Burgundy (17)
   5: Coffee (5)
   6: Copper (43)
   7: Gold (6)
   8: Green (1470)
   9: Grey (1222)
   10: Maroon (274)
   11: Multi-color (214)
   12: Navy Blue (835)
   13: Nude (115)
   14: Off White (308)
   15: Orange (283)
   16: Pink (939)
   17: Purple (956)
   18: Red (1227)
   19: Silver (1)
   20: Tan (18)
   21: Teal (89)
   22: White (2306)
   23: Yellow (567)

👔 NECKLINE (15 classes):
   0: Boat Neck (117)
   1: Collared (2128)
   2: Cowl Neck (62)
   3: Crew Neck (2615)
   4: Halter (365)
   5: High Neck (832)
   6: Keyhole (522)
   7: Mock Neck (59)
   8: Off-Shoulder (386)
   9: One-Shoulder (79)
   10: Scoop Neck (2973)
   11: Square Neck (439)
   12: Sweetheart (173)
   13: Turtleneck (53)
   14: V-Neck (5362)

📏 DRESS_LENGTH (5 classes):
   0: Floor-length (226)
   1: Knee-length (3086)
   2: Maxi (899)
   3: Midi (1005)
   4: Mini (10949)

🧵 

Cell 8: Balance Dataset & Create Train/Val/Test Split

In [9]:
# ============================================
# CELL 8: Balance & Split Dataset
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Check class distributions — balance the most critical attributes
print("📊 Checking class balance for critical attributes...")

# For neckline (most likely to be imbalanced)
MIN_NECKLINE_SAMPLES = 150
neckline_counts = clean_dress_df['neckline'].value_counts()
print(f"\nNeckline counts:")
print(neckline_counts)

# Balance neckline minority classes
balanced_dfs = []
for neckline in clean_dress_df['neckline'].unique():
    neck_df = clean_dress_df[clean_dress_df['neckline'] == neckline]
    if len(neck_df) < MIN_NECKLINE_SAMPLES:
        neck_df_up = resample(neck_df, replace=True, n_samples=MIN_NECKLINE_SAMPLES, random_state=42)
        balanced_dfs.append(neck_df_up)
    else:
        # Cap at 2500 to prevent huge class dominance
        if len(neck_df) > 2500:
            neck_df = neck_df.sample(n=2500, random_state=42)
        balanced_dfs.append(neck_df)

balanced_dress_df = pd.concat(balanced_dfs, ignore_index=True)
balanced_dress_df = balanced_dress_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n{'='*60}")
print(f"📊 AFTER BALANCING")
print(f"{'='*60}")
print(f"Total samples: {len(balanced_dress_df)}")
print(f"\nNeckline distribution (balanced):")
print(balanced_dress_df['neckline'].value_counts())

# Re-encode after balancing (indices may have changed)
for attr_name, col_name in attributes.items():
    balanced_dress_df[f'{attr_name}_encoded'] = encoders[attr_name].transform(balanced_dress_df[col_name])

# ============ TRAIN/VAL/TEST SPLIT ============
train_df, temp_df = train_test_split(
    balanced_dress_df, test_size=0.2,
    stratify=balanced_dress_df['neckline_encoded'],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    stratify=temp_df['neckline_encoded'],
    random_state=42
)

print(f"\n📊 Data Split:")
print(f"   Training:   {len(train_df)} ({len(train_df)/len(balanced_dress_df)*100:.1f}%)")
print(f"   Validation: {len(val_df)} ({len(val_df)/len(balanced_dress_df)*100:.1f}%)")
print(f"   Test:       {len(test_df)} ({len(test_df)/len(balanced_dress_df)*100:.1f}%)")

# Verify minimum 7000
if len(balanced_dress_df) >= 7000:
    print(f"\n✅ Meets 7000+ requirement: {len(balanced_dress_df)}")
else:
    print(f"\n⚠️ Only {len(balanced_dress_df)} — may need augmentation")

# Save for later use
balanced_dress_df.to_csv(os.path.join(DATA_DIR, 'dresses_balanced.csv'), index=False)
print(f"\n💾 Balanced data saved to: {DATA_DIR}/dresses_balanced.csv")

📊 Checking class balance for critical attributes...

Neckline counts:
neckline
V-Neck          5362
Scoop Neck      2973
Crew Neck       2615
Collared        2128
High Neck        832
Keyhole          522
Square Neck      439
Off-Shoulder     386
Halter           365
Sweetheart       173
Boat Neck        117
One-Shoulder      79
Cowl Neck         62
Mock Neck         59
Turtleneck        53
Name: count, dtype: int64

📊 AFTER BALANCING
Total samples: 13095

Neckline distribution (balanced):
neckline
Crew Neck       2500
V-Neck          2500
Scoop Neck      2500
Collared        2128
High Neck        832
Keyhole          522
Square Neck      439
Off-Shoulder     386
Halter           365
Sweetheart       173
Mock Neck        150
One-Shoulder     150
Cowl Neck        150
Turtleneck       150
Boat Neck        150
Name: count, dtype: int64

📊 Data Split:
   Training:   10476 (80.0%)
   Validation: 1309 (10.0%)
   Test:       1310 (10.0%)

✅ Meets 7000+ requirement: 13095

💾 Balanced data save

Cell 9: Create PyTorch Dataset & DataLoaders

In [10]:
# ============================================
# CELL 9: PyTorch Dataset & DataLoaders
# ============================================

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ============ TRANSFORMS ============
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1)
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============ DATASET CLASS ============
class DressDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        try:
            image = Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        targets = {
            'color': torch.tensor(row['color_encoded'], dtype=torch.long),
            'neckline': torch.tensor(row['neckline_encoded'], dtype=torch.long),
            'dress_length': torch.tensor(row['dress_length_encoded'], dtype=torch.long),
            'fabric': torch.tensor(row['fabric_encoded'], dtype=torch.long),
            'pattern': torch.tensor(row['pattern_encoded'], dtype=torch.long),
            'sleeve_length': torch.tensor(row['sleeve_length_encoded'], dtype=torch.long),
            'usage': torch.tensor(row['usage_encoded'], dtype=torch.long),
            'season': torch.tensor(row['season_encoded'], dtype=torch.long),
            'gender': torch.tensor(row['gender_encoded'], dtype=torch.long),
        }

        return image, targets

# ============ CREATE DATALOADERS ============
BATCH_SIZE = 32

train_dataset = DressDataset(train_df, transform=train_transform)
val_dataset = DressDataset(val_df, transform=val_transform)
test_dataset = DressDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

# Verify
images, targets = next(iter(train_loader))
print(f"🔍 Batch Verification:")
print(f"   Image batch: {images.shape}")
for key in targets:
    print(f"   {key:15s}: {targets[key][:5].tolist()}")
print(f"\n✅ DataLoaders created!")

🔍 Batch Verification:
   Image batch: torch.Size([32, 3, 224, 224])
   color          : [22, 18, 9, 17, 2]
   neckline       : [3, 14, 14, 10, 14]
   dress_length   : [4, 4, 4, 4, 1]
   fabric         : [7, 1, 1, 7, 1]
   pattern        : [0, 7, 5, 8, 0]
   sleeve_length  : [4, 2, 3, 4, 3]
   usage          : [0, 4, 0, 0, 0]
   season         : [0, 0, 2, 2, 2]
   gender         : [0, 0, 0, 0, 2]

✅ DataLoaders created!


Cell 10: Build Multi-Head ResNet-152 Dress Attribute Extractor

In [11]:
# ============================================
# CELL 10: Build Multi-Head ResNet-152 Dress Attribute Extractor
# ============================================

import torch.nn as nn
import torchvision.models as models

class DressAttributeExtractor(nn.Module):
    """
    Multi-head ResNet-152 for dress attribute extraction.
    Predicts 9 attributes simultaneously:
    color, neckline, dress_length, fabric, pattern, sleeve_length,
    usage, season, gender

    Also outputs a 512-dim feature vector for Model 3 (Fusion).
    """
    def __init__(self, num_classes_dict):
        super(DressAttributeExtractor, self).__init__()

        # Pretrained ResNet-152
        self.backbone = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V2)
        num_features = self.backbone.fc.in_features  # 2048
        self.backbone.fc = nn.Identity()

        # Shared feature layers
        self.shared_fc = nn.Sequential(
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
        )

        # Create classification head for each attribute
        self.heads = nn.ModuleDict()
        for attr_name, n_classes in num_classes_dict.items():
            if n_classes >= 10:
                # Larger head for attributes with many classes
                self.heads[attr_name] = nn.Sequential(
                    nn.Linear(512, 256),
                    nn.ReLU(inplace=True),
                    nn.Dropout(0.2),
                    nn.Linear(256, n_classes)
                )
            else:
                # Smaller head for attributes with few classes
                self.heads[attr_name] = nn.Sequential(
                    nn.Linear(512, 128),
                    nn.ReLU(inplace=True),
                    nn.Linear(128, n_classes)
                )

    def forward(self, x):
        backbone_features = self.backbone(x)
        shared = self.shared_fc(backbone_features)

        outputs = {}
        for attr_name, head in self.heads.items():
            outputs[attr_name] = head(shared)

        # Return shared features for Model 3 fusion
        outputs['features'] = shared       # 512-dim
        outputs['backbone_features'] = backbone_features  # 2048-dim

        return outputs

# ============ INITIALIZE MODEL ============
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DressAttributeExtractor(num_classes).to(device)

# Freeze backbone initially (Stage 1)
print("🔒 Freezing ResNet-152 backbone for Stage 1...")
for param in model.backbone.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\n📊 Model Summary:")
print(f"   Device: {device}")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Frozen params:    {frozen_params:,}")
print(f"\n   Architecture: ResNet-152 Multi-Head (9 heads)")
print(f"   Backbone: 2048-dim → Shared FC: 1024 → 512")
print(f"\n   Classification Heads:")
for attr_name, n_classes in num_classes.items():
    head_type = "Large (512→256→N)" if n_classes >= 10 else "Small (512→128→N)"
    print(f"   {attr_name:15s}: {head_type} → {n_classes} classes")
print(f"\n✅ Model created! (Backbone FROZEN)")

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:01<00:00, 132MB/s]


🔒 Freezing ResNet-152 backbone for Stage 1...

📊 Model Summary:
   Device: cpu
   Total params:     61,574,802
   Trainable params: 3,430,994
   Frozen params:    58,143,808

   Architecture: ResNet-152 Multi-Head (9 heads)
   Backbone: 2048-dim → Shared FC: 1024 → 512

   Classification Heads:
   color          : Large (512→256→N) → 24 classes
   neckline       : Large (512→256→N) → 15 classes
   dress_length   : Small (512→128→N) → 5 classes
   fabric         : Large (512→256→N) → 11 classes
   pattern        : Small (512→128→N) → 9 classes
   sleeve_length  : Small (512→128→N) → 6 classes
   usage          : Small (512→128→N) → 5 classes
   season         : Small (512→128→N) → 4 classes
   gender         : Small (512→128→N) → 3 classes

✅ Model created! (Backbone FROZEN)


Cell 11: Define Loss Functions, Optimizer & Scheduler

In [12]:
# ============================================
# CELL 11: Loss Functions, Optimizer & Scheduler
# ============================================

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ============ LOSS WEIGHTS (importance of each attribute) ============
LOSS_WEIGHTS = {
    'color': 2.0,           # Very important for matching
    'neckline': 1.5,        # Important for accessory selection
    'dress_length': 1.2,    # Affects accessory type
    'fabric': 1.0,          # Moderate importance
    'pattern': 1.0,         # Moderate importance
    'sleeve_length': 1.0,   # Affects bracelet/watch visibility
    'usage': 1.5,           # Important for occasion matching
    'season': 0.8,          # Less critical
    'gender': 0.8,          # Less critical (usually obvious)
}

# ============ LOSS FUNCTIONS ============
criterions = {}
for attr_name in num_classes.keys():
    criterions[attr_name] = nn.CrossEntropyLoss()

# ============ OPTIMIZER ============
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

# ============ TRAINING CONFIG ============
NUM_EPOCHS_STAGE1 = 10
NUM_EPOCHS_STAGE2 = 15
NUM_EPOCHS_TOTAL = NUM_EPOCHS_STAGE1 + NUM_EPOCHS_STAGE2
PATIENCE = 7
UNFREEZE_EPOCH = NUM_EPOCHS_STAGE1

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

print("⚙️ Training Configuration:")
print(f"   Stage 1: Epochs 1-{NUM_EPOCHS_STAGE1} (backbone FROZEN, lr=1e-3)")
print(f"   Stage 2: Epochs {NUM_EPOCHS_STAGE1+1}-{NUM_EPOCHS_TOTAL} (backbone UNFROZEN, lr=1e-5)")
print(f"   Total: {NUM_EPOCHS_TOTAL} epochs")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Early Stopping: patience={PATIENCE}")
print(f"\n   Loss Weights:")
for k, v in LOSS_WEIGHTS.items():
    print(f"      {k:15s}: {v}")
print(f"\n✅ Training configuration ready!")

⚙️ Training Configuration:
   Stage 1: Epochs 1-10 (backbone FROZEN, lr=1e-3)
   Stage 2: Epochs 11-25 (backbone UNFROZEN, lr=1e-5)
   Total: 25 epochs
   Batch Size: 32
   Early Stopping: patience=7

   Loss Weights:
      color          : 2.0
      neckline       : 1.5
      dress_length   : 1.2
      fabric         : 1.0
      pattern        : 1.0
      sleeve_length  : 1.0
      usage          : 1.5
      season         : 0.8
      gender         : 0.8

✅ Training configuration ready!


Cell 12: Training Loop with 2-Stage Fine-tuning

In [13]:
# ============================================
# CELL 12: Training Loop (2-Stage + Early Stopping)
# ============================================

from tqdm import tqdm
import time

def calculate_loss(outputs, targets):
    total_loss = 0
    for attr_name in LOSS_WEIGHTS:
        total_loss += LOSS_WEIGHTS[attr_name] * criterions[attr_name](outputs[attr_name], targets[attr_name])
    return total_loss

def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    correct = {k: 0 for k in LOSS_WEIGHTS.keys()}
    total = 0

    for images, targets in tqdm(dataloader, desc="  Training", leave=False):
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}

        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total += images.size(0)

        for key in correct:
            _, predicted = torch.max(outputs[key], 1)
            correct[key] += (predicted == targets[key]).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracies = {k: v / total * 100 for k, v in correct.items()}
    return avg_loss, accuracies

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = {k: 0 for k in LOSS_WEIGHTS.keys()}
    total = 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="  Validating", leave=False):
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}

            outputs = model(images)
            loss = calculate_loss(outputs, targets)

            total_loss += loss.item()
            total += images.size(0)

            for key in correct:
                _, predicted = torch.max(outputs[key], 1)
                correct[key] += (predicted == targets[key]).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracies = {k: v / total * 100 for k, v in correct.items()}
    return avg_loss, accuracies

# ==================== START TRAINING ====================
print("=" * 70)
print("🚀 STARTING TRAINING - Dress Attribute Extractor (ResNet-152)")
print("   2-Stage Fine-tuning | 9 Attribute Heads")
print("=" * 70)

best_val_avg_acc = 0
best_epoch = 0
patience_counter = 0
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': {k: [] for k in LOSS_WEIGHTS.keys()},
    'val_acc': {k: [] for k in LOSS_WEIGHTS.keys()},
    'lr': []
}

start_total = time.time()

for epoch in range(NUM_EPOCHS_TOTAL):
    start_time = time.time()

    # Stage 2: Unfreeze backbone
    if epoch == UNFREEZE_EPOCH:
        print(f"\n{'='*70}")
        print(f"🔓 STAGE 2: Unfreezing backbone for fine-tuning!")
        print(f"{'='*70}")

        for param in model.backbone.parameters():
            param.requires_grad = True

        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': 1e-5},
            {'params': model.shared_fc.parameters(), 'lr': 5e-5},
        ] + [
            {'params': model.heads[name].parameters(), 'lr': 5e-5}
            for name in model.heads.keys()
        ], weight_decay=1e-4)

        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Trainable: {trainable:,}\n")

    # Train & Validate
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, device)
    scheduler.step()

    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)
    for k in LOSS_WEIGHTS.keys():
        history['train_acc'][k].append(train_acc[k])
        history['val_acc'][k].append(val_acc[k])

    # Calculate average accuracy across all attributes
    val_avg_acc = np.mean(list(val_acc.values()))
    train_avg_acc = np.mean(list(train_acc.values()))

    stage = "Stage 1 (Frozen)" if epoch < UNFREEZE_EPOCH else "Stage 2 (Fine-tune)"

    # Print progress
    print(f"\n📈 Epoch [{epoch+1}/{NUM_EPOCHS_TOTAL}] - {stage} ({epoch_time:.1f}s)")
    print(f"   Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"   Train Avg Acc: {train_avg_acc:.1f}% | Val Avg Acc: {val_avg_acc:.1f}%")
    print(f"   Val Details:")
    print(f"     Color: {val_acc['color']:.1f}% | Neckline: {val_acc['neckline']:.1f}% | Length: {val_acc['dress_length']:.1f}%")
    print(f"     Fabric: {val_acc['fabric']:.1f}% | Pattern: {val_acc['pattern']:.1f}% | Sleeve: {val_acc['sleeve_length']:.1f}%")
    print(f"     Usage: {val_acc['usage']:.1f}% | Season: {val_acc['season']:.1f}% | Gender: {val_acc['gender']:.1f}%")
    print(f"   LR: {current_lr:.8f}")

    # Save best model
    if val_avg_acc > best_val_avg_acc:
        best_val_avg_acc = val_avg_acc
        best_epoch = epoch + 1
        patience_counter = 0

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_acc,
            'val_avg_accuracy': val_avg_acc,
            'num_classes': num_classes,
            'encoder_data': encoder_data,
        }, os.path.join(MODELS_DIR, 'dress_attribute_extractor_best.pth'))
        print(f"   💾 Best model saved! (Val Avg Acc: {best_val_avg_acc:.1f}%)")
    else:
        patience_counter += 1
        print(f"   ⏳ No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}!")
            break

total_time = time.time() - start_total

print(f"\n{'='*70}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best Epoch: {best_epoch}")
print(f"   Best Val Average Accuracy: {best_val_avg_acc:.1f}%")
print(f"   Total Time: {total_time/60:.1f} minutes")
print(f"{'='*70}")

🚀 STARTING TRAINING - Dress Attribute Extractor (ResNet-152)
   2-Stage Fine-tuning | 9 Attribute Heads


KeyboardInterrupt: 

Cell 13: Plot Training History

In [ ]:
# ============================================
# CELL 13: Plot Training History
# ============================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# ---- Plot 1: Loss ----
axes[0, 0].plot(history['train_loss'], label='Train', color='#2196F3', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', color='#F44336', linewidth=2)
axes[0, 0].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Unfreeze')
axes[0, 0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ---- Plot 2: Color Accuracy ----
axes[0, 1].plot(history['train_acc']['color'], label='Train', color='#2196F3', linewidth=2)
axes[0, 1].plot(history['val_acc']['color'], label='Val', color='#F44336', linewidth=2)
axes[0, 1].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7)
axes[0, 1].set_title('🎨 Color Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# ---- Plot 3: Neckline Accuracy ----
axes[0, 2].plot(history['train_acc']['neckline'], label='Train', color='#2196F3', linewidth=2)
axes[0, 2].plot(history['val_acc']['neckline'], label='Val', color='#F44336', linewidth=2)
axes[0, 2].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7)
axes[0, 2].set_title('👔 Neckline Accuracy', fontsize=13, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# ---- Plot 4: Pattern Accuracy ----
axes[1, 0].plot(history['train_acc']['pattern'], label='Train', color='#2196F3', linewidth=2)
axes[1, 0].plot(history['val_acc']['pattern'], label='Val', color='#F44336', linewidth=2)
axes[1, 0].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7)
axes[1, 0].set_title('🎭 Pattern Accuracy', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ---- Plot 5: Fabric Accuracy ----
axes[1, 1].plot(history['train_acc']['fabric'], label='Train', color='#2196F3', linewidth=2)
axes[1, 1].plot(history['val_acc']['fabric'], label='Val', color='#F44336', linewidth=2)
axes[1, 1].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7)
axes[1, 1].set_title('🧵 Fabric Accuracy', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# ---- Plot 6: All Val Accuracies Comparison ----
colors_list = ['#E91E63', '#9C27B0', '#2196F3', '#4CAF50', '#FF9800', '#795548', '#607D8B', '#FF5722', '#00BCD4']
for i, (attr_name, vals) in enumerate(history['val_acc'].items()):
    axes[1, 2].plot(vals, label=attr_name, color=colors_list[i], linewidth=1.5)
axes[1, 2].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7)
axes[1, 2].set_title('All Val Accuracies', fontsize=13, fontweight='bold')
axes[1, 2].legend(fontsize=8, ncol=2)
axes[1, 2].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Value')

plt.suptitle('Model 2: Dress Attribute Extractor — Training History',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'model2_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training history saved!")

Cell 14: Evaluate on Test Set

In [ ]:
# ============================================
# CELL 14: Evaluate on Test Set (FIXED)
# ============================================

from sklearn.metrics import classification_report
import numpy as np

# Load best model
checkpoint = torch.load(os.path.join(MODELS_DIR, 'dress_attribute_extractor_best.pth'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"📥 Loaded best model from Epoch {checkpoint['epoch']}")

# Collect predictions
all_preds = {k: [] for k in LOSS_WEIGHTS.keys()}
all_labels = {k: [] for k in LOSS_WEIGHTS.keys()}

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="🔍 Testing"):
        images = images.to(device)
        outputs = model(images)

        for key in all_preds:
            _, predicted = torch.max(outputs[key], 1)
            all_preds[key].extend(predicted.cpu().numpy())
            all_labels[key].extend(targets[key].numpy())

# ============ CLASSIFICATION REPORTS ============
key_attributes = ['color', 'neckline', 'dress_length', 'pattern', 'fabric', 'sleeve_length']

for attr in key_attributes:
    print(f"\n{'='*70}")
    print(f"📊 {attr.upper()} CLASSIFICATION REPORT")
    print(f"{'='*70}")

    # ✅ FIX: Get only labels that actually appear in test data
    unique_labels = sorted(set(all_labels[attr]) | set(all_preds[attr]))
    target_names = [encoders[attr].classes_[i] for i in unique_labels]

    print(classification_report(
        all_labels[attr], all_preds[attr],
        labels=unique_labels,
        target_names=target_names,
        digits=3, zero_division=0
    ))

# ============ OVERALL SUMMARY ============
print(f"\n{'='*70}")
print(f"📊 OVERALL TEST ACCURACY SUMMARY")
print(f"{'='*70}")
accuracies = {}
for key in all_preds:
    acc = np.mean(np.array(all_preds[key]) == np.array(all_labels[key])) * 100
    accuracies[key] = acc
    emoji_map = {
        'color': '🎨', 'neckline': '👔', 'dress_length': '📏',
        'fabric': '🧵', 'pattern': '🎭', 'sleeve_length': '💪',
        'usage': '🎯', 'season': '📅', 'gender': '👤'
    }
    emoji = emoji_map.get(key, '📋')
    print(f"   {emoji} {key:15s}: {acc:.2f}%")

avg_acc = np.mean(list(accuracies.values()))
print(f"\n   📊 AVERAGE ACCURACY: {avg_acc:.2f}%")

if avg_acc >= 75:
    print(f"\n✅ Average accuracy ({avg_acc:.1f}%) meets research target!")
else:
    print(f"\n⚠️ Average accuracy ({avg_acc:.1f}%) below 75% target.")

Cell 15: Confusion Matrices for Key Attributes

In [ ]:
# ============================================
# CELL 15: Confusion Matrices for Key Attributes
# ============================================

import seaborn as sns
from sklearn.metrics import confusion_matrix

key_attrs = ['color', 'neckline', 'dress_length', 'pattern']
fig, axes = plt.subplots(2, 2, figsize=(22, 18))

for ax, attr in zip(axes.flat, key_attrs):
    cm = confusion_matrix(all_labels[attr], all_preds[attr])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=encoders[attr].classes_,
                yticklabels=encoders[attr].classes_,
                ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(f'{attr.upper()} Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Model 2: Dress Attribute Extractor — Confusion Matrices',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'model2_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrices saved!")

Cell 16: Visual Predictions on Sample Images

In [ ]:
# ============================================
# CELL 16: Visual Predictions on Sample Images
# ============================================

import random
from PIL import Image

def predict_dress(model, image_path, device, encoders):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)

    predictions = {}
    confidences = {}

    for attr_name, enc in encoders.items():
        probs = torch.softmax(outputs[attr_name], dim=1)
        conf, pred_idx = torch.max(probs, 1)
        predictions[attr_name] = enc.classes_[pred_idx.item()]
        confidences[attr_name] = conf.item() * 100

    return predictions, confidences

# Test on 8 samples
fig, axes = plt.subplots(2, 4, figsize=(24, 14))
samples = random.sample(range(len(test_df)), 8)

for ax, idx in zip(axes.flat, samples):
    row = test_df.iloc[idx]
    img = Image.open(row['image_path']).convert('RGB')
    preds, confs = predict_dress(model, row['image_path'], device, encoders)

    ax.imshow(img)
    title = (
        f"🎨 Color: {preds['color']} ({confs['color']:.0f}%)\n"
        f"👔 Neckline: {preds['neckline']} ({confs['neckline']:.0f}%)\n"
        f"📏 Length: {preds['dress_length']} | 🧵 Fabric: {preds['fabric']}\n"
        f"🎭 Pattern: {preds['pattern']} | 💪 Sleeve: {preds['sleeve_length']}\n"
        f"🎯 Usage: {preds['usage']} | 📅 Season: {preds['season']}"
    )
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.axis('off')

plt.suptitle('Model 2: Dress Attribute Extractor — Sample Predictions',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'model2_sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample predictions saved!")

Cell 17: Save Final Model & Inference Model for Backend

In [ ]:
# ============================================
# CELL 17: Save Final Model & All Artifacts
# ============================================

import json
import pickle

# Load best model
checkpoint = torch.load(os.path.join(MODELS_DIR, 'dress_attribute_extractor_best.pth'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# ============ SAVE INFERENCE MODEL (for backend) ============
inference_save = {
    'model_state_dict': model.state_dict(),
    'num_classes': num_classes,
    'encoder_data': encoder_data,
    'input_size': [3, 224, 224],
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'attributes': list(num_classes.keys()),
}

torch.save(inference_save, os.path.join(MODELS_DIR, 'dress_attribute_extractor_inference.pth'))

# ============ SAVE MODEL INFO ============
model_info = {
    'model_name': 'DressAttributeExtractor_ResNet152',
    'architecture': 'ResNet-152 Multi-Head (2-Stage Fine-tuned)',
    'num_heads': 9,
    'attributes': {
        'color': {'num_classes': num_classes['color'], 'classes': encoder_data['color']['classes']},
        'neckline': {'num_classes': num_classes['neckline'], 'classes': encoder_data['neckline']['classes']},
        'dress_length': {'num_classes': num_classes['dress_length'], 'classes': encoder_data['dress_length']['classes']},
        'fabric': {'num_classes': num_classes['fabric'], 'classes': encoder_data['fabric']['classes']},
        'pattern': {'num_classes': num_classes['pattern'], 'classes': encoder_data['pattern']['classes']},
        'sleeve_length': {'num_classes': num_classes['sleeve_length'], 'classes': encoder_data['sleeve_length']['classes']},
        'usage': {'num_classes': num_classes['usage'], 'classes': encoder_data['usage']['classes']},
        'season': {'num_classes': num_classes['season'], 'classes': encoder_data['season']['classes']},
        'gender': {'num_classes': num_classes['gender'], 'classes': encoder_data['gender']['classes']},
    },
    'total_output_neurons': sum(num_classes.values()),
    'best_epoch': best_epoch,
    'best_val_avg_accuracy': round(best_val_avg_acc, 2),
    'training_config': {
        'stage1_epochs': NUM_EPOCHS_STAGE1,
        'stage2_epochs': NUM_EPOCHS_STAGE2,
        'batch_size': BATCH_SIZE,
        'early_stopping_patience': PATIENCE,
    },
    'dataset': {
        'name': 'Kaggle Fashion Product Images (Apparel)',
        'total_samples': len(balanced_dress_df),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
    },
    'loss_weights': LOSS_WEIGHTS,
    'pseudo_label_note': 'Neckline, fabric, pattern, sleeve_length, dress_length use pseudo-labels based on articleType heuristics + productDisplayName text mining. Color, usage, season, gender from dataset.',
}

with open(os.path.join(MODELS_DIR, 'model2_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

# Save training history
with open(os.path.join(MODELS_DIR, 'model2_history.pkl'), 'wb') as f:
    pickle.dump(history, f)

# ============ LIST ALL FILES ============
print("=" * 60)
print("📁 ALL SAVED FILES")
print("=" * 60)

for folder_name, folder_path in [('trained_models', MODELS_DIR), ('data', DATA_DIR), ('plots', PLOTS_DIR)]:
    print(f"\n📂 {folder_path}/")
    for f_name in sorted(os.listdir(folder_path)):
        size = os.path.getsize(os.path.join(folder_path, f_name))
        if size > 1024 * 1024:
            print(f"   📄 {f_name} ({size / (1024*1024):.1f} MB)")
        else:
            print(f"   📄 {f_name} ({size / 1024:.1f} KB)")